# Workshop: PCA and the geography of British dialect words

People in different parts of the United Kingdom use different words for the same everyday things. In Glasgow a small river is a *burn*; in Leeds it is a *beck*; in Bristol it is a *stream*. In this workshop you will take a table that records which words are used where, and use **principal component analysis (PCA)** to find the main patterns in it.

The interesting thing is this: we will never tell PCA where any place is. It only sees a table of numbers. Yet when we draw the results on a map, clear regions appear. PCA discovers the geography from the words alone.

You do not need to know anything about British dialects, or about British geography. Everything you need is explained as we go, and the words themselves are explained in Section 0.

### What you will practise

- centring a data matrix and computing its covariance matrix;
- finding principal components from the eigenvalues and eigenvectors of the covariance matrix;
- reading a scree plot and working out how much variance each component explains;
- telling apart **loadings** (which variables matter) and **scores** (where each data point sits);
- interpreting components, and checking whether they can be trusted;
- at the end, a short look at **independent component analysis (ICA)** and how its answer differs.

### How the workshop is organised

| Section | Topic | Rough time |
|---|---|---|
| 0 | The data: what was asked, and what the numbers mean | 10 min (reading) |
| 1 | Looking at the data | 15 min |
| 2 | PCA by hand on two variables | 20 min |
| 3 | PCA on all 139 words | 35 min |
| 4 | Can we trust it? | 25 min |
| 5 | A short look at ICA | 15 min |

Cells marked **Exercise** ask you to write or complete some code. Cells marked **Question** ask for a sentence or two in words; write your answer in the cell underneath. The questions matter as much as the code: the point of PCA is to understand the data, and a plot you cannot explain is not much use.

## 0. The data

### 0.1 Where the data come from

In 2004 and 2005 the BBC ran a large online survey called *Voices*. People were shown a list of everyday ideas, such as "tired" or "to rain lightly", and asked which word they would normally use for each one. They also gave their postcode, so we know roughly where each person lived.

Grieve, Montgomery, Nini, Murakami and Guo (2019) turned the answers into a table. For each part of the country, and for each word, the table gives the percentage of people there who chose that word. They also built a second table of the same shape from 1.8 billion words of Twitter posts sent in 2014, by counting how often each word was used in posts from each part of the country. We use the survey table for most of the workshop, and bring in the Twitter table in Section 4.

> Grieve, J., Montgomery, C., Nini, A., Murakami, A. and Guo, D. (2019). Mapping lexical dialect variation in British English using Twitter. *Frontiers in Artificial Intelligence* 2:11.

### 0.2 What was asked

The table covers 36 ideas, which we will call **concepts**. For each concept there are between two and six possible **words**. The next cell loads the full list. Read through it now; it is the key to everything that follows.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dialect_maps import plot_map, plot_rgb_map

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 50)

questions = pd.read_csv("survey_questions.csv", dtype={"concept": str})
questions

So, for example, concept **31** is "a child's soft shoes worn for PE". (PE is "physical education", the school sports lesson. The shoes are light canvas shoes with a rubber sole.) People were asked what they call these shoes, and the four answers kept in the table are *daps*, *plimsolls*, *pumps* and *trainers*.

Many of the words are ordinary English that you will know (*cold*, *baby*, *toilet*). Others are regional, and you would not be expected to know them. Here are the ones that turn out to matter most later on.

| Word | Meaning |
|---|---|
| *burn*, *beck*, *brook* | a stream (a small river) |
| *bairn*, *wean*, *babby* | a baby or small child |
| *mam*, *mom*, *mum*, *ma* | mother |
| *daps*, *pumps*, *plimsolls* | soft shoes for school sport |
| *ginnel* | an alley (a narrow path between buildings) |
| *mardy*, *stroppy* | moody, bad-tempered |
| *ned*, *chav*, *scally*, *townie* | slang, usually unkind, for a young person in cheap fashionable clothes |
| *skint*, *brassic* | having no money |
| *keks*, *trews* | trousers |
| *cack-handed*, *cag-handed* | left-handed |
| *skive*, *bunk*, *wag* | to stay away from school without permission |
| *lake* | to play (an old northern word, nothing to do with water) |

A few of the words in the survey are informal or mildly rude. They are in the data because they are the words people really use; some unfortunately are classist and there appearance in the survey may indicate that the class dynamics of the UK makes this a point of substantial variation, or perhaps the chose the researchers to use include this and not other questions is itself indicative of the complexity of class in this country.   

### 0.3 Where: postcode areas

A UK postcode looks like `BS8 1UB`. The one or two letters at the start are the **postcode area**, and each postcode area is named after its main town or city: `BS` is Bristol, `G` is Glasgow, `NE` is Newcastle, `CF` is Cardiff. There are 124 postcode areas, and they are the rows of our table. The file `areas.csv` gives the name of each one, which of the UK's nations it is in, and its position.

### 0.4 What the numbers are

Each column of the table is one word, named with its concept number, for example `31_daps`. Each entry is a percentage: of the people in that postcode area who answered the question for concept 31, what percentage said *daps*?

Because every person gives one answer, the percentages for the words of one concept add up to 100 in every area. Keep this in mind; it comes back in Section 3.

## 1. Looking at the data

Load the survey table and the table of areas.

In [ ]:
bbc = pd.read_csv("bbc_voices.csv", index_col="area")
areas = pd.read_csv("areas.csv", index_col="area")

print("survey table:", bbc.shape)
bbc.iloc[:5, :8]

In [ ]:
areas.head(8)

The column `approx_respondents` is an estimate of how many people in that area answered a typical question. We will need it in Section 4.

Here is a small helper that returns the column names for one concept, and an example of using it: the percentages for concept 31 (the shoes) in Bristol.

In [ ]:
def words_for(concept):
    # column names for one concept, e.g. words_for("31")
    return [c for c in bbc.columns if c.startswith(concept + "_")]

print(words_for("31"))
bbc.loc["BS", words_for("31")]

**Exercise 1.1.** Use `words_for` to answer these.

1. In Bristol (`BS`), what are the percentages for the words for *mother* (concept 22)? Which word is most common?
2. Do the same for Newcastle (`NE`) and Birmingham (`B`). What is different?
3. Check that the percentages for concept 22 add up to 100 in all three places.

In [ ]:
# Exercise 1.1: your code here
# Hint: bbc.loc[["BS", "NE", "B"], words_for("22")] gives all three rows at once,
# and .sum(axis=1) adds along each row.

**Question 1.1.** Which word for *mother* is most common in each of the three places?

*Your answer:*

### Drawing a map

`plot_map` colours each postcode area by a number. Give it a pandas Series indexed by postcode area. Here is the percentage of people who say *daps* for their sports shoes. (For a percentage, which is never negative, use `diverging=False`.)

In [ ]:
plot_map(bbc["31_daps"], diverging=False, title="'daps' (% of answers for concept 31)");

*Daps* is a word from Bristol, the south-west of England and south Wales. You can still hear it in Bristol.

The next cell draws all four words for this concept side by side.

In [ ]:
cols = words_for("31")
fig, axes = plt.subplots(1, len(cols), figsize=(4 * len(cols), 6))
for ax, col in zip(axes, cols):
    plot_map(bbc[col], ax=ax, diverging=False, title=col)
plt.tight_layout()

**Exercise 1.2.** Draw the same kind of figure for the words for *a stream* (concept 38), and for one other concept of your choice.

In [ ]:
# Exercise 1.2: your code here
# Hint: copy the cell above and change the concept number.

**Question 1.2.** Describe in a sentence or two where each word for *a stream* is used. Do different concepts divide the country in the same way, or in different ways?

*Your answer:*

It helps to be able to look up the areas where a word is most used. This cell lists the five areas with the highest percentage for *ginnel*, with their names.

In [ ]:
top = bbc["35_ginnel"].sort_values(ascending=False).head(5)
pd.DataFrame({"percent": top, "name": areas.loc[top.index, "name"]})

### Why we need PCA

There are 139 words, so we could draw 139 maps. That is too many to take in, and many of them would look alike: you have probably noticed already that the map for *burn* resembles the maps for other Scottish words. When many variables rise and fall together, a small number of patterns might describe most of what is going on.

That is exactly what PCA looks for. It replaces the 139 correlated variables with a few new variables, the **principal components**, chosen so that the first one captures as much of the variation in the data as any single variable can, the second captures as much as possible of what is left, and so on.

## 2. PCA by hand on two variables

Before using all 139 words we will do PCA on just two, so that we can draw everything. We take *pumps* (for sports shoes) and *ginnel* (for an alley). Each postcode area is one point in the plane.

In [ ]:
two = bbc[["31_pumps", "35_ginnel"]]
X2 = two.values            # a 124 x 2 array

plt.figure(figsize=(5.5, 5))
plt.scatter(X2[:, 0], X2[:, 1], s=15)
plt.xlabel("% saying 'pumps'")
plt.ylabel("% saying 'ginnel'")
plt.axis("equal");

The two words go together: areas where many people say *pumps* tend to be areas where many people say *ginnel*. The cloud of points is stretched along a diagonal. PCA finds that diagonal.

### The recipe

Write the data as a matrix $X$ with $n$ rows (areas) and $p$ columns (words). Here $n = 124$ and $p = 2$.

1. **Centre.** Subtract the mean of each column, so that every column has mean zero. Call the result $X_c$.
2. **Covariance matrix.** Compute
$$C = \frac{1}{n-1} X_c^{\mathsf T} X_c .$$
   This is a $p \times p$ symmetric matrix. Its diagonal entries are the variances of the columns, and its off-diagonal entries are the covariances between pairs of columns.
3. **Eigenvectors and eigenvalues.** Find the eigenvectors $v_1, v_2, \dots$ of $C$ and their eigenvalues $\lambda_1 \ge \lambda_2 \ge \dots$. The eigenvectors are the **principal components**: $v_1$ is the direction in which the data vary most, $v_2$ is the direction of greatest variation at right angles to $v_1$, and so on.
4. **Scores.** The **score** of an area on component $k$ is the position of that area along the direction $v_k$, which is $z_k = X_c v_k$. The variance of these scores is exactly $\lambda_k$.

The entries of an eigenvector are called **loadings**: they say how much each original variable contributes to that component.

**Exercise 2.1.** Carry out steps 1 and 2 for `X2`. Check your covariance matrix against `np.cov(X2, rowvar=False)`.

In [ ]:
# Exercise 2.1
n = X2.shape[0]

X2c = ...        # centre: subtract the column means (X2.mean(axis=0))
C2 = ...         # covariance matrix: X2c.T @ X2c / (n - 1)

print(C2)
print(np.cov(X2, rowvar=False))

**Question 2.1.** What do the two diagonal entries of the covariance matrix tell you? What does the sign of the off-diagonal entry tell you?

*Your answer:*

**Exercise 2.2.** Find the eigenvalues and eigenvectors of your covariance matrix with `np.linalg.eigh`. This function is designed for symmetric matrices. It returns the eigenvalues in *increasing* order, and the eigenvectors as the *columns* of a matrix, so reorder both so that the largest eigenvalue comes first. Then work out what fraction of the total variance each component explains.

In [ ]:
# Exercise 2.2
evals2, evecs2 = np.linalg.eigh(C2)

order = ...                 # indices that sort evals2 from largest to smallest
evals2 = ...                # reorder the eigenvalues
evecs2 = ...                # reorder the columns of evecs2 in the same way

print("eigenvalues:", evals2)
print("eigenvectors (columns):")
print(evecs2)
print("fraction of variance:", ...)

The next cell draws the two eigenvectors on top of the centred data. Each arrow is drawn with length equal to two standard deviations of the scores in that direction, that is $2\sqrt{\lambda_k}$.

In [ ]:
plt.figure(figsize=(5.5, 5))
plt.scatter(X2c[:, 0], X2c[:, 1], s=15, alpha=0.6)
for k, colour in enumerate(["crimson", "darkorange"]):
    v = evecs2[:, k]
    if v[np.abs(v).argmax()] < 0:      # choose which end of the line to draw (see the note below)
        v = -v
    v = v * 2 * np.sqrt(evals2[k])
    plt.annotate("", xy=v, xytext=(0, 0), annotation_clip=False,
                 arrowprops=dict(arrowstyle="->", color=colour, lw=2.5))
    plt.text(v[0] * 1.08, v[1] * 1.08, f"PC{k + 1}", color=colour, fontsize=12)
plt.xlabel("'pumps' (centred)")
plt.ylabel("'ginnel' (centred)")
plt.axis("equal");

A note on signs. If $v$ is an eigenvector then so is $-v$, so the eigenvector you computed for PC1 may point up and to the right, or down and to the left. Both are correct: the *line* is what PCA finds, and which end we call positive is arbitrary. (For the picture, the cell above always draws the end that points towards the larger percentages.) This matters whenever you interpret a component, and we deal with it properly in Section 3.

**Exercise 2.3.** Compute the scores (step 4 of the recipe), and check two facts:

1. the variance of the scores on each component equals the corresponding eigenvalue;
2. the scores on PC1 and the scores on PC2 are uncorrelated.

In [ ]:
# Exercise 2.3
Z2 = ...                    # scores: X2c @ evecs2, a 124 x 2 array

print("variance of scores:", ...)       # hint: Z2.var(axis=0, ddof=1)
print("eigenvalues:       ", evals2)
print("covariance matrix of the scores:")
print(...)                              # hint: np.cov(Z2, rowvar=False)

**Question 2.3.** (a) What fraction of the variance does PC1 explain, and what does that mean in words? (b) An area has a large score on PC1 (taking the sign for which both loadings are positive). What does that tell you about the words used there? (c) Why is it useful that the scores on different components are uncorrelated?

*Your answer:*

Finally, put the PC1 scores on the map. We have made one new variable out of two, and it has a clear geography.

In [ ]:
pc1_two_words = pd.Series(Z2[:, 0], index=bbc.index)
if evecs2[0, 0] < 0:          # choose the sign so that high = more 'pumps'
    pc1_two_words = -pc1_two_words
plot_map(pc1_two_words, title="PC1 from 'pumps' and 'ginnel'");

## 3. PCA on all 139 words

Nothing in the recipe depends on having two variables. With 139 words each area is a point in 139-dimensional space, which we cannot draw, but the covariance matrix is simply $139 \times 139$ and its eigenvectors each have 139 entries.

**Exercise 3.1.** Complete the function below so that it carries out the whole recipe. To deal with the sign problem, the function calls `fix_signs`, which flips each eigenvector so that its largest entry (in absolute value) is positive. That way everybody in the room gets the same signs.

In [ ]:
def fix_signs(evecs):
    # flip each column so that its entry of largest absolute value is positive
    biggest = np.abs(evecs).argmax(axis=0)
    signs = np.sign(evecs[biggest, np.arange(evecs.shape[1])])
    return evecs * signs

In [ ]:
# Exercise 3.1
def pca(X):
    # X is an n x p array. Returns (eigenvalues, eigenvectors, scores),
    # with the largest eigenvalue first and eigenvectors as columns.
    n = X.shape[0]
    Xc = ...                              # 1. centre
    C = ...                               # 2. covariance matrix
    evals, evecs = np.linalg.eigh(C)      # 3. eigenvalues and eigenvectors
    order = ...                           #    largest first
    evals = ...
    evecs = ...
    evecs = fix_signs(evecs)
    scores = ...                          # 4. scores
    return evals, evecs, scores

The next cell checks your function against the PCA in scikit-learn. Both lines should print `True`. If they do not, look again at your function before going on.

In [ ]:
from sklearn.decomposition import PCA

evals, evecs, Z = pca(bbc.values)
check = PCA().fit(bbc.values)

print("eigenvalues agree: ", np.allclose(evals[:20], check.explained_variance_[:20]))
print("eigenvectors agree:", np.allclose(np.abs(evecs[:, :5]), np.abs(check.components_[:5].T), atol=1e-6))

It is convenient to keep the results in labelled tables: `scores` has one row per area and one column per component, and `loadings` has one row per word.

In [ ]:
pc_names = [f"PC{k + 1}" for k in range(len(evals))]
scores = pd.DataFrame(Z, index=bbc.index, columns=pc_names)
loadings = pd.DataFrame(evecs, index=bbc.columns, columns=pc_names)

scores.iloc[:5, :4]

### 3.1 How many components matter?

The eigenvalue $\lambda_k$ is the variance of the scores on component $k$, and the eigenvalues add up to the total variance of the data. So $\lambda_k / \sum_j \lambda_j$ is the **fraction of variance explained** by component $k$. A plot of this against $k$ is called a **scree plot**.

**Exercise 3.2.** Compute the fraction of variance explained by each component, and the running total (`np.cumsum`). Plot both for the first 30 components. How many components are needed to explain 50% of the variance? 80%?

In [ ]:
# Exercise 3.2
explained = ...             # fraction of variance for each component
cumulative = ...            # running total

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
# left: scree plot of explained[:30]; right: cumulative[:30]

**Question 3.2.** (a) How much of the variance do the first two components explain together? (b) If the 139 words were completely unrelated to each other, and all varied by the same amount, what would the scree plot look like? What does the real scree plot tell you about the data?

*Your answer:*

**Question 3.3 (think about it).** The cell below counts how many eigenvalues are not zero (allowing for rounding error). The answer is less than 139. Can you explain the number? *Hint: re-read Section 0.4, and remember there are 36 concepts.*

In [ ]:
print("eigenvalues that are not (nearly) zero:", int((evals > 1e-6 * evals[0]).sum()), "out of", len(evals))

*Your answer:*

### 3.2 What do the components mean?

To interpret a component we look at two things.

- Its **loadings**: which words have large positive entries in the eigenvector, and which have large negative entries? A component is a contrast between these two groups of words.
- Its **scores**: which areas are at the positive end, and which at the negative end? Since our areas are places, we can draw the scores on a map.

Here is a helper that prints the words with the most negative and the most positive loadings.

In [ ]:
def top_words(loadings, pc, n=7):
    col = loadings[pc].sort_values()
    print(f"{pc}  most negative: " + ", ".join(f"{w} ({v:+.2f})" for w, v in col.head(n).items()))
    print(f"{pc}  most positive: " + ", ".join(f"{w} ({v:+.2f})" for w, v in col.tail(n)[::-1].items()))

top_words(loadings, "PC1")

**Exercise 3.3.** Draw maps of the scores on PC1, PC2 and PC3 side by side, and print the top words for PC2 and PC3. Put the fraction of variance explained in each map's title.

In [ ]:
# Exercise 3.3
# Hint: fig, axes = plt.subplots(1, 3, figsize=(13, 6)), then
# plot_map(scores["PC1"], ax=axes[0], title=...) and so on.

**Question 3.4.** Describe PC1 and PC2 in words. For each one: which part of the country is at the positive end, which is at the negative end, and which words characterise each end? (Use the list of concepts in Section 0 to see what the words mean.)

*Your answer:*

### 3.3 Two components at once

Each area has a score on PC1 and a score on PC2, so we can plot the areas as points in the plane of the first two components. Here they are coloured by nation, with a few areas labelled.

In [ ]:
colours = {"England": "tab:blue", "Scotland": "tab:red", "Wales": "tab:green",
           "Northern Ireland": "tab:purple", "Crown Dependency": "tab:grey"}

plt.figure(figsize=(8, 6))
for nation, colour in colours.items():
    here = areas.index[areas.nation == nation]
    plt.scatter(scores.loc[here, "PC1"], scores.loc[here, "PC2"], s=22, color=colour, label=nation)
for a in ["BS", "G", "EH", "NE", "M", "LS", "B", "L", "CF", "BT", "E", "RM", "BD", "AB", "SR", "NR"]:
    plt.annotate(areas.name[a], (scores.PC1[a], scores.PC2[a]), fontsize=7, xytext=(3, 3), textcoords="offset points")
plt.xlabel("PC1 score"); plt.ylabel("PC2 score"); plt.legend(fontsize=8);

We can also put three components on one map, by using the three scores as the amounts of red, green and blue. Areas with similar colours have similar vocabulary, as far as the first three components can tell.

In [ ]:
plot_rgb_map(scores[["PC1", "PC2", "PC3"]], title="PC1, PC2, PC3 as red, green, blue");

**Question 3.5.** Remember that PCA was given only a table of percentages. It was told nothing about where the areas are, or which are next to which. Why, then, do neighbouring areas come out with similar scores?

*Your answer:*

## 4. Can we trust it?

PCA will always return components, whatever data you give it. A careful analyst asks whether the components reflect something real, or something accidental about this particular table. We look at two checks.

### 4.1 What is going on in PC3?

PC1 and PC2 gave tidy maps. PC3 did not. Let us see which areas have the most extreme scores on PC3 and PC4.

**Exercise 4.1.** For PC3, find the three areas with the lowest scores and the three with the highest. Look them up in `areas`, and note the column `approx_respondents`. Do the same for PC4.

In [ ]:
# Exercise 4.1
# Hint: scores["PC3"].sort_values() puts the areas in order; .index[:3] and
# .index[-3:] give the two ends; areas.loc[...] looks them up.

**Question 4.1.** What do the most extreme areas have in common? Explain why PCA gives them so much weight. *Hint: if only three people in an area answered a question, what values can a percentage take?*

*Your answer:*

### 4.2 Removing the smallest samples

**Exercise 4.2.** Make a new table `bbc_big` containing only the areas with at least 15 respondents. Run your `pca` on it, and build `scores_big`, `loadings_big` and `explained_big` in the same way as before. How many areas are left?

In [ ]:
# Exercise 4.2
keep = ...                  # index of areas with approx_respondents >= 15
bbc_big = ...               # the rows of bbc for those areas

evals_big, evecs_big, Z_big = ...
scores_big = pd.DataFrame(Z_big, index=bbc_big.index, columns=pc_names)
loadings_big = pd.DataFrame(evecs_big, index=bbc.columns, columns=pc_names)
explained_big = ...

print(len(bbc_big), "areas kept")

The next cell compares the new components with the old ones. For each of the first four components it prints the correlation between the old scores and the new scores, over the areas that are in both. A value near 1 (or $-1$) means the component has hardly changed.

In [ ]:
for pc in ["PC1", "PC2", "PC3", "PC4"]:
    r = np.corrcoef(scores.loc[keep, pc], scores_big[pc])[0, 1]
    print(f"{pc}: correlation between old and new scores = {r:+.2f}")

**Exercise 4.3.** Draw the maps of the first three new components, and print their top words. (Areas that are not in the table you pass to `plot_map` are drawn in grey, so the areas we removed will show up grey.)

In [ ]:
# Exercise 4.3: your code here (reuse your code from Exercise 3.3)

**Question 4.2.** (a) Which components survived the removal of the small areas, and which did not? (b) Describe the new PC3. (c) What general lesson about PCA do you take from this?

*Your answer:*

### 4.3 A second opinion from different data

A stronger check is to repeat the analysis on data collected in a completely different way. The file `twitter.csv` has the same rows and columns as the survey table, but the percentages come from counting words in Twitter posts sent from each area in 2014, about ten years after the survey. Nobody was asked a question; people were simply writing.

**Exercise 4.4.** Load `twitter.csv`, keep the same areas as `bbc_big`, and run your `pca`. Then, for each of the first three components, compute the correlation between the survey scores and the Twitter scores. Draw the Twitter PC1 and PC2 maps.

In [ ]:
# Exercise 4.4
twitter = pd.read_csv("twitter.csv", index_col="area")
tw_big = ...                # same areas as bbc_big

evals_tw, evecs_tw, Z_tw = ...
scores_tw = pd.DataFrame(Z_tw, index=tw_big.index, columns=pc_names)

# correlations between scores_big and scores_tw for PC1, PC2, PC3, then the maps

**Question 4.3.** What do the correlations tell you? (The sign of a correlation here means nothing, because the sign of a component is arbitrary; look at its size.) Why is this a more convincing check than, say, running PCA on the survey data a second time?

*Your answer:*

## 5. A short look at ICA

### What PCA cannot do

Look back at your maps. PC1 says "Scotland versus everywhere else". PC2 says "the north of England versus the south-east". These are broad contrasts: every area in the country has a place somewhere along each one.

Now recall what you saw in Section 1. *Daps* belongs to Bristol and south Wales. *Mam* belongs to the north-east. *Mom* belongs to the area around Birmingham. These are compact regions, each with a few words of its own. Where are they in the PCA? They are there, but spread across many components and mixed together, for two reasons.

- PCA orders its components by variance, so each one tries to explain as much as it can about *all* the areas at once.
- Each component must be at right angles to all the earlier ones. After PC1 has taken "Scotland", PC2 is not free to describe a region however it likes; it has to fit around PC1. By PC5 or PC6 the components are complicated mixtures, positive in two or three unrelated places and negative in others.

### What ICA does differently

**Independent component analysis** also describes each area as a combination of a few components. The difference is in what it asks of them. PCA asks for scores that are uncorrelated and have the largest possible variance. ICA asks for scores that are as **statistically independent** of each other as possible, which is a stronger requirement than being uncorrelated.

In practice, ICA finds independent components by looking for directions in which the scores are as far from a bell-shaped (Gaussian) distribution as possible. One very non-Gaussian shape is this: most areas have a score near zero, and a few areas have a large score. A component like that describes something that is present in a few places and absent everywhere else, which is exactly what a regional dialect is.

The usual procedure, and the one used below, has two steps. First use PCA to reduce the data to a small number of components (here six). Then, within that six-dimensional space, rotate the axes to make the scores as non-Gaussian as possible. So ICA does not find a different six-dimensional space from PCA. It finds a different, and often more meaningful, set of axes inside the same space.

The code for this section is given; your job is to interpret the result.

In [ ]:
from sklearn.decomposition import FastICA

n_ic = 6
ica = FastICA(n_components=n_ic, whiten="unit-variance", random_state=0, max_iter=10000)
S = ica.fit_transform(bbc_big.values - bbc_big.values.mean(axis=0))     # scores: areas x 6
A = ica.mixing_                                                         # loadings: words x 6

# as with PCA the sign is arbitrary: make the largest score of each component positive
signs = np.sign(S[np.abs(S).argmax(axis=0), np.arange(n_ic)])
S, A = S * signs, A * signs

ic_names = [f"IC{k + 1}" for k in range(n_ic)]
ic_scores = pd.DataFrame(S, index=bbc_big.index, columns=ic_names)
ic_loadings = pd.DataFrame(A, index=bbc.columns, columns=ic_names)

fig, axes = plt.subplots(1, n_ic, figsize=(3.6 * n_ic, 5.5))
for ic, ax in zip(ic_names, axes):
    plot_map(ic_scores[ic], ax=ax, title=ic, colorbar=False)
plt.tight_layout()

In [ ]:
for ic in ic_names:
    top_areas = ic_scores[ic].sort_values(ascending=False).index[:6]
    top = ic_loadings[ic].sort_values(ascending=False).index[:6]
    print(f"{ic}  areas: " + ", ".join(areas.name[a] for a in top_areas))
    print(f"     words: " + ", ".join(top), "\n")

**Question 5.1.** For each independent component, name the region it picks out and two or three of its words. (The order of the components is arbitrary, and may differ from your neighbour's.) Is Bristol in any of them?

*Your answer:*

The difference between the two methods shows up clearly in the *distribution* of the scores. The next cell draws histograms of the scores for the first two principal components (blue) and for the six independent components (red).

In [ ]:
from scipy.stats import kurtosis

fig, axes = plt.subplots(2, 4, figsize=(15, 5.5))
panels = [("PC1", scores_big.PC1), ("PC2", scores_big.PC2)] + [(ic, ic_scores[ic]) for ic in ic_names]
for ax, (name, values) in zip(axes.ravel(), panels):
    ax.hist((values - values.mean()) / values.std(), bins=25, color="tab:blue" if name.startswith("PC") else "tab:red")
    ax.set_title(f"{name}: excess kurtosis {kurtosis(values):.1f}", fontsize=10)
    ax.set_xlabel("score (standardised)")
plt.tight_layout()

**Excess kurtosis** measures how heavy the tails of a distribution are compared with a Gaussian, for which it is zero. A large positive value means "mostly near zero, with a few large values".

**Question 5.2.** (a) Describe the difference between the histograms of the principal components and those of the independent components. How does this fit with what each method is looking for? (b) In Section 4 we removed the areas with tiny samples before running ICA. Predict what ICA would find if we had left them in. If you have time, test your prediction by running the ICA cell on `bbc` in place of `bbc_big`.

*Your answer:*

## Summary

- PCA replaces many correlated variables by a few uncorrelated ones. The components are the eigenvectors of the covariance matrix, and the eigenvalues say how much variance each one carries.
- **Loadings** tell you which variables make up a component; **scores** tell you where each data point sits on it. You need both to interpret a component.
- The sign of a component is arbitrary.
- Constraints in the data (here, percentages adding up to 100) show up as zero eigenvalues.
- PCA finds variance, whether it is signal or noise. Check which data points drive a component, and where you can, check against independent data.
- ICA works inside the space found by PCA, and rotates the axes to make the scores as non-Gaussian as possible. Here that turns broad contrasts into compact regions.

## If you have time

1. **Standardising.** We did PCA on the covariance matrix, so words with large variance count for more. Divide each column of `bbc_big` by its standard deviation before calling `pca` (this is PCA on the *correlation* matrix). Do PC1 and PC2 change? Which words gain influence, and is that a good thing?
2. **Reconstruction.** Using only the first $k$ components, the data can be approximately rebuilt as $\hat X = \bar x + Z_k V_k^{\mathsf T}$, where $Z_k$ holds the first $k$ columns of scores and $V_k$ the first $k$ eigenvectors. Rebuild the map of `31_daps` with $k = 2$, $6$ and $20$. How many components are needed before the Bristol region appears?
3. **How many independent components?** Rerun the ICA with 4 and with 10 components. Which regions are found every time, and which come and go?
4. **Where do you fit?** Pick an area you know, or the area you live in now, and look up its PCA and ICA scores. Which words does it share with its neighbours?